In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"

## Analysis for Damaged Park Vehicle cases (H 7 - LLM 5 and H 5 - LLM 5 cases)

In [ ]:
# Data Paths
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"
translated_unfalltext_table_path = "data/translated_unfalltext_0_103111.csv"
few_shot_llm_predicted_labels_table_path = "data/few_shot_llm_predicted_labels_unfalltext_0_103111.csv"
damaged_parked_vehicle_case_llm_labels_path = "data/damaged_park_prompt_llm_predicted_labels_unfalltext_0_10000.csv"
beteiligte_table_path = "data/D_Beteiligte_RData.csv"

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)
translated_unfalltext_table = pd.read_csv(translated_unfalltext_table_path)
few_shot_llm_predicted_labels_table = pd.read_csv(few_shot_llm_predicted_labels_table_path)
damaged_parked_vehicle_case_llm_labels = pd.read_csv(damaged_parked_vehicle_case_llm_labels_path)
beteiligte_table = pd.read_csv(beteiligte_table_path)

In [ ]:
beteiligte_table.BT_BETEIL_K.unique()

In [ ]:
# Group by UN_KEY and count the number of Sonstige und unbekannte Fahrzeuge (Unfallflucht) in BT_BETEIL_K field

# Group by UN_KEY and count occurrences of 'Sonstige und unbekannte Fahrzeuge (Unfallflucht)'
unknown_vehicle_count_by_accident = beteiligte_table.groupby('UN_KEY')['BT_BETEIL_K'].apply(
    lambda x: (x == 'Sonstige und unbekannte Fahrzeuge (Unfallflucht)').sum()
).reset_index(name='Unknown (Hit and Run) Vehicle Count')

unknown_vehicle_count_by_accident["Is Unknown (Hit and Run) Vehicle Involved"] = unknown_vehicle_count_by_accident[
    'Unknown (Hit and Run) Vehicle Count'] > 0

unknown_vehicle_count_by_accident.drop(columns='Unknown (Hit and Run) Vehicle Count', inplace=True)

In [ ]:
unknown_vehicle_count_by_accident.head(3)

In [ ]:
# Merge the two tables on UN_KEY
unfall_text_table_labels_merged = unfall_table.merge(
    unfalltext_table, on="UN_KEY", how="left").merge(
        few_shot_llm_predicted_labels_table[["UN_KEY", "few_shot_LLM_Labels"]], on="UN_KEY", how="left").merge(
            translated_unfalltext_table[["UN_KEY", "Translated_Text"]], on="UN_KEY", how="left").merge(
                damaged_parked_vehicle_case_llm_labels[["UN_KEY", "damaged_park_prompt_LLM_Labels"]], on="UN_KEY", how="left").merge(
                    unknown_vehicle_count_by_accident, on="UN_KEY", how="left")

# Remove the rows not existing in unfalltext_table (so they don't have accident text descriptions)
unfall_text_table_labels_merged = unfall_text_table_labels_merged[unfall_text_table_labels_merged.UN_KEY.isin(unfalltext_table.UN_KEY)]

# Date columns
unfall_text_table_labels_merged["DateTime"] = pd.to_datetime(unfall_text_table_labels_merged["DateTime"], errors="coerce")
unfall_text_table_labels_merged["Year"] = unfall_text_table_labels_merged["DateTime"].dt.year
unfall_text_table_labels_merged["Month"] = unfall_text_table_labels_merged["DateTime"].dt.month
unfall_text_table_labels_merged["Year_Month"] = unfall_text_table_labels_merged["DateTime"].dt.to_period("M")

unfall_text_table_labels_merged.shape

In [ ]:
# Remove duplicated text list from Sascha

# Read the duplicated text description list
duplicated_texts = pd.read_excel("/home/accidents/data/contributions_wise2425/duplicated_text_description_list.xlsx")
duplicated_texts_to_remove = duplicated_texts[duplicated_texts["filter"]=="yes"]["Text"].tolist()

# Remove the duplicated text descriptions
unfall_text_table_labels_merged = unfall_text_table_labels_merged[~unfall_text_table_labels_merged["Text"].isin(duplicated_texts_to_remove)]
unfall_text_table_labels_merged.shape

In [ ]:
unfall_text_table_labels_merged.head(3)

In [ ]:
# Human Label 5 or LLM Few Shot Label 5 case count (48% of all accidents)
unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp==5) | (unfall_text_table_labels_merged.few_shot_LLM_Labels==5)].shape

In [ ]:
# Write the case to the csv
# unfall_text_table_labels_merged[(unfall_text_table_labels_merged.Unfalltyp==5) | (unfall_text_table_labels_merged.few_shot_LLM_Labels==5)].to_csv("data/human_label_5_or_few_shot_label_5_cases.csv", index=False)

In [ ]:
unfall_text_table_labels_merged.damaged_park_prompt_LLM_Labels.value_counts(dropna=False)

In [ ]:
unfall_text_table_labels_merged[
    (unfall_text_table_labels_merged.Unfalltyp == 5) & 
    (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)
].damaged_park_prompt_LLM_Labels.value_counts()

In [ ]:
unfall_text_table_labels_merged[
    (unfall_text_table_labels_merged.Unfalltyp == 7) & 
    (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)
].damaged_park_prompt_LLM_Labels.value_counts()

In [ ]:
print("Human 5 & LLM 5 Case")

# Definitions
definitions = {
    1: "Conflict between moving and parking/parked vehicle",
    2: "Conflict between two parking/parked vehicles",
    3: "The parked vehicle was damaged by an unknown source"
}

# Filter dataset
filtered_df = unfall_text_table_labels_merged[
    (unfall_text_table_labels_merged.Unfalltyp == 5) & 
    (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)
]

# Compute normalized value counts
value_counts = filtered_df.damaged_park_prompt_LLM_Labels.value_counts(normalize=True)

# Compute hit-and-run ratio per category
hit_and_run_ratios = filtered_df.groupby("damaged_park_prompt_LLM_Labels")[
    "Is Unknown (Hit and Run) Vehicle Involved"
].mean()
# Compute overall hit-and-run ratio
overall_hit_and_run_ratio = filtered_df["Is Unknown (Hit and Run) Vehicle Involved"].mean() * 100
# Print overall hit-and-run ratio
print(f"Overall Hit-and-run Vehicle Ratio: {overall_hit_and_run_ratio:.2f}%")



# Print results
for key, definition in definitions.items():
    percentage = value_counts.get(key, 0) * 100  # Get percentage or default to 0 if key is missing
    hit_and_run_ratio = hit_and_run_ratios.get(key, 0) * 100  # Get ratio or default to 0 if key is missing
    print(f"{key}: {definition} ({percentage:.2f}%) | Hit-and-run Vehicle: {hit_and_run_ratio:.2f}%")

In [ ]:
print("Human 7 & LLM 5 Case")

# Definitions
definitions = {
    1: "Conflict between moving and parking/parked vehicle",
    2: "Conflict between two parking/parked vehicles",
    3: "The parked vehicle was damaged by an unknown source"
}

# Filter dataset
filtered_df = unfall_text_table_labels_merged[
    (unfall_text_table_labels_merged.Unfalltyp == 7) & 
    (unfall_text_table_labels_merged.few_shot_LLM_Labels == 5)
]

# Compute normalized value counts
value_counts = filtered_df.damaged_park_prompt_LLM_Labels.value_counts(normalize=True)

# Compute hit-and-run ratio per category
hit_and_run_ratios = filtered_df.groupby("damaged_park_prompt_LLM_Labels")[
    "Is Unknown (Hit and Run) Vehicle Involved"
].mean()
# Compute overall hit-and-run ratio
overall_hit_and_run_ratio = filtered_df["Is Unknown (Hit and Run) Vehicle Involved"].mean() * 100
# Print overall hit-and-run ratio
print(f"Overall Hit-and-run Vehicle Ratio: {overall_hit_and_run_ratio:.2f}%")



# Print results
for key, definition in definitions.items():
    percentage = value_counts.get(key, 0) * 100  # Get percentage or default to 0 if key is missing
    hit_and_run_ratio = hit_and_run_ratios.get(key, 0) * 100  # Get ratio or default to 0 if key is missing
    print(f"{key}: {definition} ({percentage:.2f}%) | Hit-and-run Vehicle: {hit_and_run_ratio:.2f}%")